# Experiment 07: EE-TransUNet + CLAHE on G1020 Dataset

**Date:** 2025-10-08  
**Author:** Sebastian Siedler  
**Objective:** Combine Vision Transformer with CLAHE preprocessing and larger G1020 dataset

---

## 📋 Experiment Overview

### Hypothesis
By adding CLAHE preprocessing and training on 78% more data (G1020: 714 samples vs REFUGE: 400), we expect:
1. **Better edge detection** from CLAHE contrast enhancement
2. **Improved generalization** from larger dataset
3. **Reduced overfitting** (train-val gap < 5%)
4. **Higher performance** (88-90% Dice vs 87.2% in Exp 06)

### Key Changes from Experiment 06
1. ✅ **CLAHE Preprocessing** (LAB mode, clip_limit=2.0)
2. ✅ **G1020 Dataset** (714 train vs 400 REFUGE)
3. ✅ **Lower Learning Rate** (0.001 vs 0.01)
4. ✅ **Adam Optimizer** (vs SGD)
5. ✅ **Larger Image Size** (430×430 vs 224×224)

### Configuration
- **Model:** EE-TransUNet ViT-Tiny (5.7M params)
- **Preprocessing:** CLAHE (LAB mode)
- **Dataset:** G1020 (cropped ROI)
- **Training samples:** 714
- **Validation samples:** 153
- **Test samples:** 153

---

## 1️⃣ Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import json
from datetime import datetime

# Project imports
from models.ee_transunet import VisionTransformer
from models.configs import get_tiny_config
from data_loader.dataset import RetinaDataset, RetinaDatasetTest
from utils.metrics import batch_metrics, calculate_cdr

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU - Training will be slow!")

## 2️⃣ Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Model
    'model': 'EE-TransUNet ViT-Tiny + CLAHE',
    'model_name': 'ViT-Tiny',
    'img_size': 430,  # Native G1020 size
    'patch_size': 16,
    'n_classes': 2,
    'n_skip': 0,  # No skip connections
    
    # Data - G1020 Dataset
    'data_dir': '../../datasets/G1020',
    'train_csv': '../../datasets/G1020/train.csv',
    'val_csv': '../../datasets/G1020/val.csv',
    'test_csv': '../../datasets/G1020/test.csv',
    'use_cropped': True,  # G1020 already cropped
    'use_clahe': True,    # ✅ ENABLE CLAHE PREPROCESSING
    
    # Training
    'epochs': 50,
    'batch_size': 4,  # Reduced due to larger image size (430×430)
    'learning_rate': 0.001,  # Reduced from 0.01 (lesson from Exp 06!)
    'weight_decay': 1e-4,
    'num_workers': 4,
    
    # Output
    'output_dir': './results',
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/visualizations", exist_ok=True)

# Save configuration
with open(f"{CONFIG['output_dir']}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=4)

print("✅ Configuration saved")
print(json.dumps(CONFIG, indent=2))

## 3️⃣ Data Loading with CLAHE

In [ ]:
# Create datasets
print("Loading G1020 datasets with CLAHE preprocessing...")

train_dataset = RetinaDataset(
    csv_file=CONFIG['train_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],  # ✅ CLAHE enabled
)

val_dataset = RetinaDataset(
    csv_file=CONFIG['val_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],  # ✅ CLAHE enabled
)

test_dataset = RetinaDatasetTest(
    csv_file=CONFIG['test_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],  # ✅ CLAHE enabled
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=CONFIG['num_workers'], pin_memory=True)

print(f"✅ G1020 Dataset loaded with CLAHE")
print(f"   Train samples: {len(train_dataset)} (70%)")
print(f"   Validation samples: {len(val_dataset)} (15%)")
print(f"   Test samples: {len(test_dataset)} (15%)")
print(f"   Data-to-param ratio: {len(train_dataset)*1000/5674114:.1f} samples per 1000 params")

### 🔍 Visualize CLAHE Effect

In [ ]:
# Visualize CLAHE preprocessing on a sample
sample_idx = 0
image, mask = train_dataset[sample_idx]

# Denormalize image
img_np = image.numpy().transpose(1, 2, 0)
img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img_np = np.clip(img_np, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_np)
axes[0].set_title('CLAHE Enhanced Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mask[0], cmap='gray')
axes[1].set_title('Optic Disc Mask', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mask[1], cmap='gray')
axes[2].set_title('Optic Cup Mask', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.suptitle('G1020 Sample with CLAHE Preprocessing', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ CLAHE preprocessing is working correctly")

## 4️⃣ Model Definition

In [ ]:
# Load model configuration
config_vit = get_tiny_config()
config_vit.n_classes = CONFIG['n_classes']
config_vit.n_skip = CONFIG['n_skip']

# Create model
model = VisionTransformer(config_vit, img_size=CONFIG['img_size'], num_classes=CONFIG['n_classes']).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ EE-TransUNet ViT-Tiny model created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"\n   Model config:")
print(f"      - Input size: {CONFIG['img_size']}×{CONFIG['img_size']}")
print(f"      - Hidden size: {config_vit.hidden_size}")
print(f"      - MLP dim: {config_vit.transformer.mlp_dim}")
print(f"      - Num heads: {config_vit.transformer.num_heads}")
print(f"      - Num layers: {config_vit.transformer.num_layers}")
print(f"      - Patch size: {config_vit.patches.size}")
print(f"      - Patches per image: {(CONFIG['img_size']//16)**2}")

In [ ]:
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'],
                       weight_decay=CONFIG['weight_decay'])

print("✅ Loss function: BCEWithLogitsLoss")
print("✅ Optimizer: Adam (more stable than SGD)")
print(f"✅ Learning rate: {CONFIG['learning_rate']} (10× lower than Exp 06)")

## 5️⃣ Training

In [ ]:
# Training functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    epoch_dice = 0
    
    pbar = tqdm(loader, desc='Training')
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return epoch_loss / len(loader), epoch_dice / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    epoch_dice_disc = 0
    epoch_dice_cup = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
            epoch_dice_disc += metrics['dice_disc']
            epoch_dice_cup += metrics['dice_cup']
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return {
        'loss': epoch_loss / len(loader),
        'dice_mean': epoch_dice / len(loader),
        'dice_disc': epoch_dice_disc / len(loader),
        'dice_cup': epoch_dice_cup / len(loader),
    }

print("✅ Training functions defined")

In [ ]:
# Training loop
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [],
           'val_dice_disc': [], 'val_dice_cup': []}
best_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15
start_time = datetime.now()

print(f"\n{'='*60}")
print("Starting training on G1020 dataset...")
print(f"{'='*60}\n")
print(f"⚙️  Configuration:")
print(f"   Dataset: G1020 (714 train, 153 val)")
print(f"   Preprocessing: CLAHE (LAB mode)")
print(f"   Model: ViT-Tiny (5.7M params)")
print(f"   Optimizer: Adam (LR={CONFIG['learning_rate']})")
print(f"   Early stopping: {early_stop_patience} epochs")
print(f"{'='*60}\n")

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 60)
    
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_metrics['loss'])
    history['val_dice'].append(val_metrics['dice_mean'])
    history['val_dice_disc'].append(val_metrics['dice_disc'])
    history['val_dice_cup'].append(val_metrics['dice_cup'])
    
    overfitting_gap = train_dice - val_metrics['dice_mean']
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"  Val Loss:   {val_metrics['loss']:.4f} | Val Dice:   {val_metrics['dice_mean']:.4f}")
    print(f"  Val Disc:   {val_metrics['dice_disc']:.4f} | Val Cup:    {val_metrics['dice_cup']:.4f}")
    print(f"  Overfitting Gap: {overfitting_gap:.4f} ({'✅' if overfitting_gap < 0.05 else '⚠️'})")
    
    if val_metrics['dice_mean'] > best_dice:
        best_dice = val_metrics['dice_mean']
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'args': CONFIG,
        }, f"{CONFIG['output_dir']}/best_model.pth")
        print(f"  ✅ Best model saved (Dice: {best_dice:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  📊 No improvement ({epochs_without_improvement}/{early_stop_patience})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'args': CONFIG,
        }, f"{CONFIG['output_dir']}/checkpoint_epoch_{epoch+1}.pth")
        print(f"  💾 Checkpoint saved (epoch {epoch+1})")
    
    # Early stopping
    if epochs_without_improvement >= early_stop_patience:
        print(f"\n⏹️  Early stopping triggered (no improvement for {early_stop_patience} epochs)")
        break

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds() / 3600

print(f"\n{'='*60}")
print(f"Training completed in {training_time:.2f} hours")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"Final epoch: {epoch+1}/{CONFIG['epochs']}")
print(f"{'='*60}")

### 📈 Training Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_dice'], label='Train Dice', linewidth=2)
axes[1].plot(history['val_dice'], label='Val Dice (Overall)', linewidth=2)
axes[1].plot(history['val_dice_disc'], label='Val Dice (Disc)', linewidth=2, linestyle='--')
axes[1].plot(history['val_dice_cup'], label='Val Dice (Cup)', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Training and Validation Dice Scores', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('EE-TransUNet + CLAHE on G1020 Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

with open(f"{CONFIG['output_dir']}/training_history.json", 'w') as f:
    json.dump(history, f, indent=4)

print("✅ Training curves saved to results/training_curves.png")
print("✅ Training history saved to results/training_history.json")

## 6️⃣ Testing

In [ ]:
# Load best model
print("Loading best model...")
checkpoint = torch.load(f"{CONFIG['output_dir']}/best_model.pth", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best model loaded (from epoch {checkpoint['epoch']+1})")
print(f"   Validation Dice: {checkpoint['metrics']['dice_mean']:.4f}")
print(f"   Validation Disc: {checkpoint['metrics']['dice_disc']:.4f}")
print(f"   Validation Cup:  {checkpoint['metrics']['dice_cup']:.4f}")

In [ ]:
# Test model on G1020 test set
print("\nTesting model on G1020 test set...")

test_metrics = {'dice_mean': [], 'dice_disc': [], 'dice_cup': [], 'cdr_mae': []}
all_predictions = []
all_masks = []
all_images = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing')
    for images, masks in pbar:
        images_device = images.to(device)
        masks_device = masks.to(device)
        
        outputs = model(images_device)
        metrics = batch_metrics(outputs, masks_device)
        
        test_metrics['dice_mean'].append(metrics['dice_mean'])
        test_metrics['dice_disc'].append(metrics['dice_disc'])
        test_metrics['dice_cup'].append(metrics['dice_cup'])
        test_metrics['cdr_mae'].append(metrics['cdr_mae'])
        
        preds = torch.sigmoid(outputs).cpu()
        all_predictions.append(preds)
        all_masks.append(masks.cpu())
        all_images.append(images.cpu())
        
        pbar.set_postfix({'dice': f"{metrics['dice_mean']:.4f}", 'cdr_mae': f"{metrics['cdr_mae']:.4f}"})

# Calculate overall test results
test_results = {
    'dice_mean': np.mean(test_metrics['dice_mean']),
    'dice_std': np.std(test_metrics['dice_mean']),
    'dice_disc': np.mean(test_metrics['dice_disc']),
    'dice_cup': np.mean(test_metrics['dice_cup']),
    'cdr_mae': np.mean(test_metrics['cdr_mae']),
    'cdr_std': np.std(test_metrics['cdr_mae']),
}

print("\n" + "="*60)
print("TEST RESULTS - G1020 Dataset")
print("="*60)
print(f"Overall Dice Score:     {test_results['dice_mean']:.4f} ± {test_results['dice_std']:.4f} ({test_results['dice_mean']*100:.2f}%)")
print(f"Disc Dice Score:        {test_results['dice_disc']:.4f} ({test_results['dice_disc']*100:.2f}%)")
print(f"Cup Dice Score:         {test_results['dice_cup']:.4f} ({test_results['dice_cup']*100:.2f}%)")
print(f"CDR Mean Absolute Error: {test_results['cdr_mae']:.4f} ± {test_results['cdr_std']:.4f}")
print("="*60)

# Compare with Experiment 06 (REFUGE without CLAHE)
exp06_dice = 0.8721
improvement = (test_results['dice_mean'] - exp06_dice) * 100
print(f"\n📊 Comparison with Experiment 06:")
print(f"   Exp 06 (REFUGE, no CLAHE): {exp06_dice:.4f} ({exp06_dice*100:.2f}%)")
print(f"   Exp 07 (G1020 + CLAHE):    {test_results['dice_mean']:.4f} ({test_results['dice_mean']*100:.2f}%)")
print(f"   Improvement: {improvement:+.2f}%")

with open(f"{CONFIG['output_dir']}/test_results.json", 'w') as f:
    json.dump(test_results, f, indent=4)

print("\n✅ Test results saved to results/test_results.json")

## 7️⃣ Visualizations

In [ ]:
# Concatenate predictions
all_predictions = torch.cat(all_predictions, dim=0)
all_masks = torch.cat(all_masks, dim=0)
all_images = torch.cat(all_images, dim=0)

# Calculate Dice for all samples
dice_scores = []
for i in range(len(all_predictions)):
    disc_pred = (all_predictions[i, 0].numpy() > 0.5).astype(float)
    cup_pred = (all_predictions[i, 1].numpy() > 0.5).astype(float)
    disc_gt = all_masks[i, 0].numpy()
    cup_gt = all_masks[i, 1].numpy()
    
    disc_dice = 2 * (disc_pred * disc_gt).sum() / (disc_pred.sum() + disc_gt.sum() + 1e-8)
    cup_dice = 2 * (cup_pred * cup_gt).sum() / (cup_pred.sum() + cup_gt.sum() + 1e-8)
    overall_dice = (disc_dice + cup_dice) / 2
    dice_scores.append(overall_dice)

dice_scores = np.array(dice_scores)
best_idx = np.argmax(dice_scores)
worst_idx = np.argmin(dice_scores)
median_indices = np.argsort(dice_scores)[len(dice_scores)//2-1:len(dice_scores)//2+1]

vis_indices = [best_idx, median_indices[0], median_indices[1], worst_idx]

print(f"Selected samples for visualization:")
print(f"  Best:   Sample {best_idx} (Dice: {dice_scores[best_idx]:.3f})")
print(f"  Median: Sample {median_indices[0]} (Dice: {dice_scores[median_indices[0]]:.3f})")
print(f"  Median: Sample {median_indices[1]} (Dice: {dice_scores[median_indices[1]]:.3f})")
print(f"  Worst:  Sample {worst_idx} (Dice: {dice_scores[worst_idx]:.3f})")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(4, 5, figsize=(20, 16))

for i, idx in enumerate(vis_indices):
    image = all_images[idx]
    mask_gt = all_masks[idx]
    pred = all_predictions[idx]
    
    # Denormalize image
    img_np = image.numpy().transpose(1, 2, 0)
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    disc_gt = mask_gt[0].numpy()
    cup_gt = mask_gt[1].numpy()
    disc_pred = (pred[0].numpy() > 0.5).astype(float)
    cup_pred = (pred[1].numpy() > 0.5).astype(float)
    
    disc_dice = 2 * (disc_pred * disc_gt).sum() / (disc_pred.sum() + disc_gt.sum() + 1e-8)
    cup_dice = 2 * (cup_pred * cup_gt).sum() / (cup_pred.sum() + cup_gt.sum() + 1e-8)
    overall_dice = (disc_dice + cup_dice) / 2
    
    cdr_gt = cup_gt.sum() / (disc_gt.sum() + 1e-8)
    cdr_pred = cup_pred.sum() / (disc_pred.sum() + 1e-8)
    
    sample_label = ['Best', 'Median', 'Median', 'Worst'][i]
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'{sample_label} Sample {idx}\nOverall Dice: {overall_dice:.3f}', fontweight='bold')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(disc_gt, cmap='gray')
    axes[i, 1].set_title(f'GT Disc\n(CDR: {cdr_gt:.3f})')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(cup_gt, cmap='gray')
    axes[i, 2].set_title('GT Cup')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(disc_pred, cmap='gray')
    axes[i, 3].set_title(f'Pred Disc\nDice: {disc_dice:.3f}')
    axes[i, 3].axis('off')
    
    axes[i, 4].imshow(cup_pred, cmap='gray')
    axes[i, 4].set_title(f'Pred Cup\nDice: {cup_dice:.3f}\nCDR: {cdr_pred:.3f}')
    axes[i, 4].axis('off')

plt.suptitle('EE-TransUNet + CLAHE - G1020 Test Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/visualizations/predictions.png", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualizations saved to results/visualizations/predictions.png")

## 8️⃣ Results Summary & Analysis

### 🎯 Performance Analysis

Run the cell below after testing completes to see detailed analysis.

In [ ]:
# Summary statistics
print("="*70)
print("EXPERIMENT 07 FINAL SUMMARY")
print("="*70)
print(f"\n📊 Model: EE-TransUNet ViT-Tiny + CLAHE")
print(f"   Parameters: {total_params:,} (5.7M)")
print(f"   Dataset: G1020 (714 train, 153 val, 153 test)")
print(f"   Preprocessing: CLAHE (LAB mode)")

print(f"\n📈 Training Results:")
print(f"   Best validation Dice: {best_dice:.4f} ({best_dice*100:.2f}%)")
print(f"   Training epochs: {len(history['train_dice'])}")
print(f"   Training time: {training_time:.2f} hours")
print(f"   Final overfitting gap: {(history['train_dice'][-1] - history['val_dice'][-1]):.4f}")

print(f"\n🎯 Test Results:")
print(f"   Overall Dice: {test_results['dice_mean']:.4f} ± {test_results['dice_std']:.4f}")
print(f"   Disc Dice:    {test_results['dice_disc']:.4f}")
print(f"   Cup Dice:     {test_results['dice_cup']:.4f}")
print(f"   CDR MAE:      {test_results['cdr_mae']:.4f} ± {test_results['cdr_std']:.4f}")

print(f"\n📊 Comparison with Experiment 06 (REFUGE without CLAHE):")
exp06_results = {'dice': 0.8721, 'disc': 0.9397, 'cup': 0.8045, 'cdr': 0.0869}
print(f"   Overall Dice: Exp06={exp06_results['dice']:.4f} | Exp07={test_results['dice_mean']:.4f} | Δ={test_results['dice_mean']-exp06_results['dice']:+.4f}")
print(f"   Disc Dice:    Exp06={exp06_results['disc']:.4f} | Exp07={test_results['dice_disc']:.4f} | Δ={test_results['dice_disc']-exp06_results['disc']:+.4f}")
print(f"   Cup Dice:     Exp06={exp06_results['cup']:.4f} | Exp07={test_results['dice_cup']:.4f} | Δ={test_results['dice_cup']-exp06_results['cup']:+.4f}")
print(f"   CDR MAE:      Exp06={exp06_results['cdr']:.4f} | Exp07={test_results['cdr_mae']:.4f} | Δ={test_results['cdr_mae']-exp06_results['cdr']:+.4f}")

print(f"\n✅ Key Findings:")
if test_results['dice_mean'] > exp06_results['dice']:
    print(f"   🏆 CLAHE + G1020 improved performance by {(test_results['dice_mean']-exp06_results['dice'])*100:+.2f}%")
else:
    print(f"   ⚠️ Performance similar or lower than Exp06 ({(test_results['dice_mean']-exp06_results['dice'])*100:+.2f}%)")

overfitting_gap = history['train_dice'][-1] - history['val_dice'][-1]
if overfitting_gap < 0.05:
    print(f"   ✅ Low overfitting gap ({overfitting_gap:.4f} < 0.05)")
else:
    print(f"   ⚠️ Significant overfitting ({overfitting_gap:.4f} ≥ 0.05) - consider augmentation")

if test_results['dice_cup'] > exp06_results['cup']:
    print(f"   ✅ CLAHE improved cup segmentation ({test_results['dice_cup']:.4f} vs {exp06_results['cup']:.4f})")

print("\n" + "="*70)

## 🎯 Conclusions & Next Steps

### Expected Outcomes
- **If Dice > 88%:** CLAHE + G1020 combination is successful → Add augmentation to push to 90%
- **If Dice ~87-88%:** Similar to Exp06 → CLAHE benefits offset by dataset differences
- **If Dice < 87%:** Lower than Exp06 → G1020 may have different characteristics or lower quality labels

### Recommendations
1. **If successful:** Try ViT-Small with more capacity
2. **If similar:** Add data augmentation (rotations, flips, elastic)
3. **If worse:** Analyze G1020 data quality and consider hybrid dataset

### Files Generated
- `results/best_model.pth` - Best model checkpoint
- `results/training_history.json` - Training metrics
- `results/training_curves.png` - Loss/Dice plots
- `results/test_results.json` - Test metrics
- `results/visualizations/predictions.png` - Sample visualizations

---

**Experiment Status:** ✅ Complete (after running all cells)